In [70]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import plotly.express as px

In [71]:
df = pd.read_csv("Filtered dataset/Training_data.csv")
display(df.head())
print(df["fog_label"].value_counts())

,timestamp_utc,rh_sensor,temp_sensor,pm1,pm25,pm10,Temperature_C,Dew_Point_C,Humidity_%,Speed_kmh,Pressure_hPa,Precip_Rate_mm,Precip_Accum_mm,Wind,tmpc_ASOS,dewpoint_ASOS,visibility_km_mean,fog_label
0,2023-07-10 00:15:00+00:00,82.54,23.2,2.63,3.11,7.98,21.11,19.56,91.0,10.46,1011.18,0.0,0.0,East,21.11,20.0,52.31,0
1,2023-07-10 00:20:00+00:00,82.72,23.2,2.57,3.01,6.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.11,20.0,50.39,0
2,2023-07-10 00:25:00+00:00,82.72,23.2,2.60,3.00,6.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.11,20.0,51.90,0
3,2023-07-10 00:30:00+00:00,82.62,23.2,2.55,3.01,12.23,21.28,19.56,90.0,5.47,1011.18,0.0,0.0,SSE,21.11,20.0,52.27,0
4,2023-07-10 00:35:00+00:00,82.76,23.2,2.94,3.51,8.62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.11,20.0,53.01,0


fog_label
0    245982
1      1119
Name: count, dtype: int64


In [72]:
total_nan_count = df.isna().sum().sum()
print(f"Total number of NaN values in the entire DataFrame: {total_nan_count}")
print("\nNaN values per column:")
print(df.isna().sum())

Total number of NaN values in the entire DataFrame: 313370

NaN values per column:
timestamp_utc             0
rh_sensor             20052
temp_sensor           20052
pm1                   20405
pm25                  20405
pm10                  20405
Temperature_C         14434
Dew_Point_C           14434
Humidity_%            14434
Speed_kmh              6692
Pressure_hPa           6692
Precip_Rate_mm         6692
Precip_Accum_mm        6692
Wind                  10269
tmpc_ASOS             44240
dewpoint_ASOS         44458
visibility_km_mean    43014
fog_label                 0
dtype: int64


In [73]:
df_clean = df.copy()
total_nan_count = df_clean.isna().sum().sum()
print(f"Total number of NaN values in the entire DataFrame: {total_nan_count}")
print("\nNaN values per column:")
print(df_clean.isna().sum())

Total number of NaN values in the entire DataFrame: 313370

NaN values per column:
timestamp_utc             0
rh_sensor             20052
temp_sensor           20052
pm1                   20405
pm25                  20405
pm10                  20405
Temperature_C         14434
Dew_Point_C           14434
Humidity_%            14434
Speed_kmh              6692
Pressure_hPa           6692
Precip_Rate_mm         6692
Precip_Accum_mm        6692
Wind                  10269
tmpc_ASOS             44240
dewpoint_ASOS         44458
visibility_km_mean    43014
fog_label                 0
dtype: int64


In [74]:
df_clean['Wind'] = df_clean['Wind'].str.upper()
print(df['Wind'].value_counts())

Wind
ENE      28148
East     26563
SW       21537
SSW      21462
ESE      20191
NE       18257
NW       15864
South    13820
SE       13265
WNW      11382
SSE       9524
NNW       8959
WSW       8951
NNE       8383
West      6406
North     4120
Name: count, dtype: int64


In [75]:
wind_mapping = {
    'NORTH': 'N',
    'SOUTH': 'S',
    'EAST': 'E',
    'WEST': 'W', 
} 

df_clean['Wind'] = df_clean['Wind'].replace(wind_mapping)
print(df_clean['Wind'].value_counts())

Wind
ENE    28148
E      26563
SW     21537
SSW    21462
ESE    20191
NE     18257
NW     15864
S      13820
SE     13265
WNW    11382
SSE     9524
NNW     8959
WSW     8951
NNE     8383
W       6406
N       4120
Name: count, dtype: int64


In [76]:
feature_cols = [
    "rh_sensor",
    "temp_sensor",
    "pm1",
    "pm25",
    "pm10",
    "Temperature_C",
    "Dew_Point_C",
    "Humidity_%",
    "Speed_kmh",
    "Pressure_hPa",
    "Precip_Rate_mm",
    "Precip_Accum_mm",
]

X_raw = df[feature_cols].dropna(how='any')
y = df.loc[X_raw.index, "fog_label"]
print(X_raw.shape, y.shape)

(213909, 12) (213909,)


In [77]:
scaler = MinMaxScaler()
X = scaler.fit_transform(X_raw)
X[:1]

array([[0.83614124, 0.63055062, 0.0161644 , 0.01757066, 0.00107101,
        0.65048544, 0.8593786 , 0.9       , 0.12265478, 0.47247366,
        0.        , 0.        ]])

GMM train and test

In [78]:
gmm_fog = GaussianMixture(n_components=2,covariance_type="full",random_state=42)
gmm_fog.fit(X)

clusters = gmm_fog.predict(X)
cluster0_mean = y[clusters == 0].mean()
cluster1_mean = y[clusters == 1].mean()
print("Cluster 0 mean fog_label:", cluster0_mean)
print("Cluster 1 mean fog_label:", cluster1_mean)

# Cluster with higher fog_label frequency = fog cluster
fog_cluster = 0 if cluster0_mean > cluster1_mean else 1
print("Chosen cluster for fog is cluster", fog_cluster) # fog_cluster
np.unique(clusters, return_counts=True)

Cluster 0 mean fog_label: 0.019308746862328634
Cluster 1 mean fog_label: 0.0011237432605291156
Chosen cluster for fog is cluster 0


(array([0, 1], dtype=int64), array([ 46611, 167298], dtype=int64))

Feature visualization

In [ ]:
df_feat_vis = X_raw[feature_cols].copy()
df_feat_vis["cluster"] = clusters

n_features = len(feature_cols)
n_cols = 3
n_rows = int(np.ceil(n_features/n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(feature_cols):
    ax = axes[i]
    sns.kdeplot(
        data=df_feat_vis,
        x=feat,
        hue="cluster",
        common_norm=False,
        fill=True,
        alpha=0.4,
        ax=ax,
    )
    ax.set_title(feat)
    ax.set_xlabel("")
    ax.grid(True, alpha=0.3)

# Hide any unused subplots if n_features is not a multiple of n_cols
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Feature distributions by GMM cluster", fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
y_pred = (clusters == fog_cluster).astype(int)

cm = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[-1, 1])
disp.plot()
plt.title(f"GMM confusion matrix")
plt.show()

print(classification_report(y, y_pred, digits=3))

In [ ]:
f1 = "Humidity_%"
f2 = "pm10"

x1 = X_raw[f1]
x2 = X_raw[f2]

fig, ax = plt.subplots(figsize=(6, 5))

# background: all points by cluster
for k in np.unique(clusters):
    mask = clusters == k
    ax.scatter(x1[mask],x2[mask],s=8,alpha=0.2,label=f"Cluster {k}")

# overlay: true fog points
fog_mask = (y == 1)
ax.scatter(x1[fog_mask],x2[fog_mask],s=20,marker="x",color="green",label="True fog (label=1)",alpha=0.1)

ax.set_xlabel(f1)
ax.set_ylabel(f2)
ax.set_title("Clusters with true fog points overlaid")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
means_scaled = gmm_fog.means_ # shape (n_components, n_features)
means_orig   = scaler.inverse_transform(means_scaled)

f1_idx = feature_cols.index(f1)
f2_idx = feature_cols.index(f2)

means_orig_f1 = means_orig[:, f1_idx]
means_orig_f2 = means_orig[:, f2_idx]

fig, ax = plt.subplots(figsize=(6, 5))

# data points (light background)
ax.scatter(X_raw[f1], X_raw[f2], s=5, alpha=0.2, label="Data")

# component centers
for k in range(means_orig.shape[0]):
    ax.scatter(means_orig_f1[k],means_orig_f2[k],s=80,marker="D",label=f"Component {k} mean")

ax.set_xlabel(f1)
ax.set_ylabel(f2)
ax.set_title("GMM component centers in feature space")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
gmm_proba = gmm_fog.predict_proba(X)[:, fog_cluster]

fog_proba_series = pd.Series(gmm_proba, index=X_raw.index)
label_series = y  # from earlier

fig, ax = plt.subplots(figsize=(6, 4))

# non-fog
ax.hist(fog_proba_series[label_series == 0],bins=20,alpha=0.6,label="True no fog (label=0)")

# fog
ax.hist(fog_proba_series[label_series == 1],bins=20,alpha=0.6,label="True fog (label=1)")

ax.set_xlabel("GMM fog probability")
ax.set_ylabel("Count")
ax.set_title("Distribution of GMM fog probability by true label")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
df_pm3d = X_raw[["pm1", "pm25", "pm10"]].copy()
df_pm3d["cluster"] = y_pred

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(df_pm3d["pm1"],df_pm3d["pm25"],df_pm3d["pm10"],
                     c=df_pm3d["cluster"], cmap='viridis',alpha=0.6)

ax.set_xlabel('PM1')
ax.set_ylabel('PM2.5')
ax.set_zlabel('PM10')
ax.set_title('3D Cluster Visualization')
plt.colorbar(scatter, label='Cluster')
plt.show()